# 02 HadUK-Grid access and download size

Lists the archive, works out what the download costs, and verifies access by
pulling one file.

Directory listings are public and carry file sizes. File downloads redirect to a
login page unless a CEDA access token is sent as a bearer header. Create a token
at https://services.ceda.ac.uk/account/token/. The first cell asks for it if
`CEDA_ACCESS_TOKEN` is not already set, and holds it for this kernel session
only.

In [ ]:
import pandas as pd

from fdh import weather as wx
from fdh.config import load_config, require

config = load_config()
BASE = require(config.weather.base_url, "weather.base_url")
print("token set:", wx.ensure_token())

## What the archive holds

In [ ]:
wx.list_directories(BASE + "/")

In [ ]:
daily = sorted(name for name, has_day in wx.daily_variables(BASE).items() if has_day)
{name: wx.list_directories(f"{BASE}/{name}/day/") for name in daily}

## Files and size for the weather window

The window runs from `weather.window_start` to `weather.release_covers_to` in
config.yaml. The annual release covers to 31 December 2025; anything later comes
from the Met Office provisional product, which is a different file with extra
compression.

In [ ]:
VARIABLES = require(config.weather.variables, "weather.variables")
VERSION = require(config.weather.version_directory, "weather.version_directory")
START = config.weather.window_start
END = config.weather.release_covers_to

files = {name: wx.daily_files(BASE, name, VERSION, START, END) for name in VARIABLES}
summary = pd.DataFrame(
    [
        {
            "variable": name,
            "files": len(items),
            "gigabytes": round(wx.total_bytes(items) / 1e9, 1),
            "first": items[0].start,
            "last": items[-1].end,
        }
        for name, items in files.items()
    ]
)
summary.loc[len(summary)] = ["total", summary.files.sum(), summary.gigabytes.sum(), None, None]
summary

## Two ways to take it

Mirroring keeps every NetCDF file on disk. Streaming downloads one month,
reduces it to service area means, and discards the file, which turns tens of
gigabytes into a table of a few hundred kilobytes. 

In [ ]:
print(wx.wget_command(f"{BASE}/rainfall/day/{VERSION}/", "data/raw/weather"))

## Verify access on one file

In [ ]:
probe = files["rainfall"][0]
probe.name, probe.url, round(probe.size / 1e6, 1)

In [ ]:
path = wx.download(probe, config.paths.resolved("raw_dir") / "weather" / "probe")
print(path, round(path.stat().st_size / 1e6, 1), "MB")

## Inspect it

Dimensions, variables and time span of the probe file.

In [ ]:
from netCDF4 import Dataset, num2date

with Dataset(path) as ds:
    print({name: len(dim) for name, dim in ds.dimensions.items()})
    for name, var in ds.variables.items():
        print(f"{name}: {var.dimensions} {var.dtype}")
    t = ds.variables["time"]
    print(num2date(t[0], t.units), "to", num2date(t[-1], t.units))